# status

> Workflow status and lifecycle route handlers

In [ ]:
#| default_exp routes.core.status

In [ ]:
#| export
from typing import Tuple, Dict, Callable

from fasthtml.common import APIRouter

from cjm_fasthtml_interactions.core.state_store import get_session_id

from cjm_fasthtml_workflow_transcript_decomp.workflow.workflow import StructureDecompWorkflow

## Handlers

In [ ]:
#| export
def _handle_current_status(
    workflow: StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess  # FastHTML session object
):  # Appropriate UI component based on current state
    """Return current workflow status - determines what to show."""
    session_id = get_session_id(sess)
    
    # Check for in-progress workflow (via server-side state store)
    workflow_state = workflow.state_store.get_state(workflow.config.workflow_id, session_id)
    current_step = workflow.state_store.get_current_step(workflow.config.workflow_id, session_id)
    
    # Restore external paths from state to SourceService (fixes persistence across restarts)
    step_states = workflow_state.get("step_states", {})
    selection_state = step_states.get("selection", {})
    external_db_paths = selection_state.get("external_db_paths", [])
    if external_db_paths:
        workflow.source_service.set_external_paths(external_db_paths)
    
    # If there's existing state, resume the workflow
    if current_step or workflow_state:
        return workflow._stepflow_router.start(request, sess)
    
    # Start fresh
    return workflow._stepflow_router.start(request, sess)

In [ ]:
#| export
def _handle_reset(
    workflow: StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess  # FastHTML session object
):  # StepFlow start view
    """Reset workflow and return to start."""
    session_id = get_session_id(sess)
    workflow.state_store.clear_state(workflow.config.workflow_id, session_id)
    return workflow._stepflow_router.start(request, sess)

## Router

In [ ]:
#| export
def init_status_router(
    workflow: StructureDecompWorkflow,  # The workflow instance
    prefix: str,  # Route prefix (e.g., "/workflow/core/status")
) -> Tuple[APIRouter, Dict[str, Callable]]:  # (router, route_dict)
    """Initialize status and lifecycle routes."""
    router = APIRouter(prefix=prefix)

    @router
    def current_status(request, sess):
        """Get current workflow status."""
        return _handle_current_status(workflow, request, sess)

    @router
    def reset(request, sess):
        """Reset workflow to beginning."""
        return _handle_reset(workflow, request, sess)

    routes = {
        "current_status": current_status,
        "reset": reset,
    }

    return router, routes

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()